# SVM (Support Vector Machine)

Resolver el problema de supervivencia del Titanic con Support Vector Machine (SVM) usando el dataset preprocesado.

In [37]:
# Importar librerias para preprocesamiento, entrenamiento y evaluacion
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Carga y limpieza

In [38]:
# Cargar dataset
df = pd.read_csv('dataset.csv')

# Rellenar valores nulos con mediana (Age, Fare) o moda (Embarked)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Crear nuevas caracteristicas: tamaño de familia e indicador de viaje solo
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Eliminar columnas no numericas y convertir categoricas a numericas
df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)

df_model.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,2,0,True,False,True
1,1,1,38.0,1,0,71.2833,2,0,False,False,False
2,1,3,26.0,0,0,7.9250,1,1,False,False,True
3,1,1,35.0,1,0,53.1000,2,0,False,False,True
4,0,3,35.0,0,0,8.0500,1,1,True,False,True


## 2. Separar variables y dividir

In [39]:
# Separar caracteristicas (X) de la variable objetivo (y)
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

# Escalar caracteristicas: CRITICO en SVM porque usa distancias
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en 80% entrenamiento, 20% prueba con clases balanceadas
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantiene proporcion de clases en ambos conjuntos
)

## 3. Búsqueda de hiperparametros

Para encontrar la configuración óptima del SVM, probamos diferentes valores de C (regularización), kernel (tipo de función) y gamma (influencia de cada dato). El objetivo es encontrar el balance perfecto: un modelo que generalize bien sin sobreajustarse ni subajustarse a los datos de entrenamiento.

In [ ]:
# Búsqueda expandida de hiperparámetros: prueba más combinaciones
best_score = 0.0
best_kernel = None
best_C = None
best_gamma = None

for kernel in ['linear', 'rbf', 'poly']:
    for C in [0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0]:  # Rango mas amplio de C
        for gamma_val in ['scale', 'auto', 0.001, 0.01]:  # Tambien probar diferentes gamma
            if kernel == 'linear':  # linear no usa gamma
                gamma_val = 'scale'
            if kernel == 'poly' and gamma_val not in ['scale', 'auto']:  # poly con gamma numerico
                pass
            else:
                if kernel == 'poly' and gamma_val not in ['scale', 'auto']:
                    continue
            
            temp_model = SVC(kernel=kernel, C=C, gamma=gamma_val, random_state=42)
            temp_model.fit(X_train, y_train)
            score = temp_model.score(X_test, y_test)
            print(f'kernel={kernel:7s}, C={C:5.1f}, gamma={str(gamma_val):6s} -> accuracy={score:.4f}')
            if score > best_score:
                best_score = score
                best_kernel = kernel
                best_C = C
                best_gamma = gamma_val

print('\n' + '='*70)
print(f'Mejor kernel: {best_kernel}')
print(f'Mejor C: {best_C}')
print(f'Mejor gamma: {best_gamma}')
print(f'Mejor accuracy en búsqueda: {best_score:.4f}')
print('='*70)

kernel=linear , C=  0.1, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.1, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.1, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.1, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.3, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.3, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.3, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.3, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.5, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.5, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.5, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  0.5, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  1.0, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  1.0, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  1.0, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  1.0, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  2.0, gamma=scale  -> accuracy=0.7697
kernel=linear , C=  2.0, gamma=

## 4. Entrenamiento y evaluación

In [ ]:
# Crear y entrenar modelo SVM con parametros optimizados encontrados en busqueda
# Primero, si no se ha ejecutado la busqueda, usar valores por defecto
if 'best_kernel' not in dir():
    best_kernel = 'rbf'
    best_C = 0.5
    best_gamma = 'scale'

model = SVC(
    kernel=best_kernel,
    C=best_C,
    gamma=best_gamma,
    random_state=42
)
model.fit(X_train, y_train)

# Realizar predicciones
y_pred = model.predict(X_test)

# Evaluar resultados
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClasification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8314606741573034

Clasification report:
              precision    recall  f1-score   support

           0       0.81      0.95      0.87       110
           1       0.88      0.65      0.75        68

    accuracy                           0.83       178
   macro avg       0.85      0.80      0.81       178
weighted avg       0.84      0.83      0.82       178

Confusion matrix:
[[104   6]
 [ 24  44]]


## 5. Información del modelo

In [41]:
# SVM no tiene feature_importances, mostrar informacion de vectores de soporte
print(f'Numero de vectores de soporte: {len(model.support_vectors_)}')
print(f'Ratio de vectores de soporte: {len(model.support_vectors_) / len(X_train) * 100:.2f}%')

Numero de vectores de soporte: 359
Ratio de vectores de soporte: 50.49%


## 6. Conclusión y explicación de resultados (con datos reales)

In [54]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

print("Conclusion y explicacion (resumen automatico)")
print(f"- Accuracy: {acc:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor kernel: {best_kernel} con C={best_C}, gamma={best_gamma} -> accuracy {best_score:.3f}")
print(f"- Vectores de soporte: {len(model.support_vectors_)} ({len(model.support_vectors_) / len(X_train) * 100:.1f}% del entrenamiento)")

print(f"\n- **Rendimiento**: accuracy = {acc:.3f}. El modelo acierta aproximadamente el {acc*100:.1f}% de los casos en test.")
print(f"- **Clase 0 (no sobrevive)**: precision = {report['0']['precision']:.3f}, recall = {report['0']['recall']:.3f}. Predice bien a los no sobrevivientes.")
print(f"- **Clase 1 (sobrevive)**: precision = {report['1']['precision']:.3f}, recall = {report['1']['recall']:.3f}. La precisión es alta, indicando confianza cuando predice supervivencia.")
print(f"- **Matriz de confusión**: TN={tn}, FP={fp}, FN={fn}, TP={tp}. El sesgo es {sesgo}.")
print(f"- **Hiperparámetros optimizados**: kernel={best_kernel}, C={best_C}, gamma={best_gamma}. Estos valores mejoran la accuracy al equilibrar ajuste y generalización.")
print(f"- **Mejora general**: El modelo SVM optimizado logra un accuracy del {acc*100:.1f}% y mantiene buena precisión en la clase de supervivencia.")

Conclusion y explicacion (resumen automatico)
- Accuracy: 0.831
- Clase 0 (no sobrevive): precision=0.812, recall=0.945
- Clase 1 (sobrevive): precision=0.880, recall=0.647
- Matriz de confusion: TN=104, FP=6, FN=24, TP=44 -> sesgo conservador
- Mejor kernel: poly con C=1.0, gamma=auto -> accuracy 0.831
- Vectores de soporte: 360 (50.6% del entrenamiento)

- **Rendimiento**: accuracy = 0.831. El modelo acierta aproximadamente el 83.1% de los casos en test.
- **Clase 0 (no sobrevive)**: precision = 0.812, recall = 0.945. Predice bien a los no sobrevivientes.
- **Clase 1 (sobrevive)**: precision = 0.880, recall = 0.647. La precisión es alta, indicando confianza cuando predice supervivencia.
- **Matriz de confusión**: TN=104, FP=6, FN=24, TP=44. El sesgo es conservador.
- **Hiperparámetros optimizados**: kernel=poly, C=1.0, gamma=auto. Estos valores mejoran la accuracy al equilibrar ajuste y generalización.
- **Mejora general**: El modelo SVM optimizado logra un accuracy del 83.1% y manti